# 13 The human review stage: who reads what, and how many

The cascade from notebook 05 ends in a person for the messages nobody else can settle, and notebook 12 showed that a failed tier sends more of them there. Review time is the scarcest budget in the system: Meridian's release checklist caps it at 5% of messages. This notebook answers four questions a team must settle before the queue exists:

1. If a reviewer can read only a few messages, which ones should they be?
2. How many of the messages the machine acted on must someone read to learn that the machine's error rate is under 5%?
3. When a new build runs in shadow beside the live one, which messages need a human label?
4. What must the review tool enforce so that a verdict is the reviewer's own and a batch is not silently incomplete?

**What the numbers cover.** The 200 Meridian test messages, the bge-small head with temperature scaling and the conformal gate from notebooks 03 and 05, and the recorded slow-tier answers. The gate's temperature and threshold were fit on these same messages in notebook 03, so the errors counted below are in-sample for the gate. The final cascade makes 5 errors in 200, and every count here rests on those 5, so read the orderings as an illustration of how to measure, not as rates to plan around. No person is involved: the protocol in Part 4 is run by a script that plays a reviewer who always matches the gold label, to show the mechanics and not the behavior of real reviewers.

In [1]:
import json
import numpy as np
from pathlib import Path
from scipy.special import softmax
from scipy.stats import binomtest, hypergeom
from sklearn.linear_model import LogisticRegression
import meridian as m

RESULTS = Path("data/results")
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
r03 = json.loads((RESULTS / "03_calibration_and_conformal.json").read_text())
ENC, T, ALPHA = r03["encoder"], r03["T"], r03["alpha_chosen"]
Q = r03[f"conformal_alpha_{ALPHA}"]["q"]

tr, te = m.load("train"), m.load("test")
_, ytr = m.texts_labels(tr); _, yte = m.texts_labels(te)
E = m.cached_embeddings(ENC)
clf = LogisticRegression(C=r01[ENC]["C"], max_iter=3000).fit(E["train"], ytr)
P = softmax(clf.decision_function(E["test"]) / T, axis=1)
top, conf = P.argmax(1), P.max(1)
acted = np.array([(p >= 1 - Q).sum() == 1 for p in P])        # the gate keeps exactly one route
idmap = json.loads(Path("data/llm/test_id_map.json").read_text())
ans = {idmap[j["id"]]: j["route"] for j in map(json.loads, Path("data/llm/test_haiku_answers.jsonl").read_text().splitlines())}
slow = np.array([m.ROUTE_ID[ans[r["id"]]] for r in te])
final = np.where(acted, top, slow)                              # the cascade's answer: encoder if the gate acts, else the slow tier
wrong = final != yte
n = len(te)
print(f"{n} messages | gate acts on {acted.sum()} | the cascade is wrong on {wrong.sum()}: {int((wrong & acted).sum())} among acted, {int((wrong & ~acted).sum())} among the {int((~acted).sum())} escalated")

200 messages | gate acts on 183 | the cascade is wrong on 5: 3 among acted, 2 among the 17 escalated


**Goal.** Fix the stream the reviewer sees: 200 messages, of which the cascade gets 5 wrong, 3 of them among the 183 the encoder acted on alone and 2 among the 17 that went to the slow tier.

## Part 1. If a reviewer can read only k messages, which k?

A reviewer reads in some order. The yield of a reading order is the number of the cascade's errors found among the first k messages. Two orders are compared with random order. **Lowest confidence first** sorts all messages by the encoder's top probability, smallest first. **Escalated first** reads the 17 messages the gate escalated, then the rest by confidence. Random order is averaged over 2,000 shuffles.

In [2]:
rng = np.random.default_rng(0)
orders = {"lowest confidence first": np.argsort(conf),
          "escalated first, then lowest confidence": np.lexsort((conf, acted))}
print(f"{'messages read':>14s}" + "".join(f"{k:>42s}" for k in orders) + f"{'random order (mean)':>24s}")
yield_rows = {}
for k in (10, 17, 30, 50):
    found = {name: int(wrong[o[:k]].sum()) for name, o in orders.items()}
    rand = float(np.mean([wrong[rng.permutation(n)[:k]].sum() for _ in range(2000)]))
    yield_rows[k] = {**found, "random": rand}
    print(f"{k:>10d} ({k / n:>4.0%})" + "".join(f"{found[nm]:>42d}" for nm in orders) + f"{rand:>24.2f}")
print(f"\nerrors to find: {int(wrong.sum())}")

 messages read                   lowest confidence first   escalated first, then lowest confidence     random order (mean)
        10 (  5%)                                         0                                         0                    0.26
        17 (  8%)                                         2                                         2                    0.43
        30 ( 15%)                                         4                                         4                    0.74
        50 ( 25%)                                         5                                         5                    1.26

errors to find: 5


**Goal.** For four reading budgets, count how many of the 5 errors each order finds.

**What we got.** Reading the 30 lowest-confidence messages (15% of the stream) finds 4 of the 5 errors, where random order would find 0.74 on average, and reading 50 (25%) finds all 5, against 1.26 for random order. Reading only 10 finds none: the first ten messages by confidence are ones the cascade got right. Both orders give the same counts here: 15 of the 17 escalated messages are among the 17 least confident.

**What this shows for the pattern.** A reading order that puts the model's doubt first multiplies what a review hour finds, and the gain is measurable on any labeled set with one line of code. Five errors cannot support a rate, and the in-sample gate flatters the ordering. On your own system, run this on the labeled set before sizing the queue, and report errors found per 100 messages read next to the queue size.

## Part 2. How many acted messages must someone read?

The encoder acted alone on 183 messages and was wrong on 3 (1.6%, Wilson interval 0.6% to 4.7%). A team that wants to confirm in production that the unreviewed share stays under 5% errors reads a random sample of the acted messages. Reading is the cost, so the number to know is how many reads give an answer.

With a fixed number of errors in the acted set, the chance that a random sample of k messages contains at least one is a hypergeometric probability. The second column below is the upper end of the Wilson interval if the audit finds zero errors: the largest error rate the audit cannot rule out.

In [3]:
na, ea = int(acted.sum()), int((wrong & acted).sum())
lo, hi = m.wilson(ea, na)
print(f"acted {na}, wrong {ea} ({ea / na:.1%}), Wilson [{lo:.3f}, {hi:.3f}]\n")
print(f"{'messages read':>14s}{'P(read at least 1 error)':>28s}{'upper bound if 0 errors seen':>32s}")
audit_rows = {}
for k in (20, 40, 75, 100, na):
    p_any = float(1 - hypergeom.cdf(0, na, ea, k)); _, up = m.wilson(0, k)
    audit_rows[k] = {"p_any": p_any, "upper_if_zero": up}
    print(f"{k:>14d}{p_any:>28.1%}{up:>32.3f}")
k = 1
while m.wilson(0, k)[1] >= 0.05: k += 1
print(f"\nsmallest sample with zero errors that shows the error rate is below 5%: {k} messages")

acted 183, wrong 3 (1.6%), Wilson [0.006, 0.047]

 messages read    P(read at least 1 error)    upper bound if 0 errors seen
            20                       29.5%                           0.161
            40                       52.5%                           0.088
            75                       79.7%                           0.049
           100                       90.9%                           0.037
           183                      100.0%                           0.021

smallest sample with zero errors that shows the error rate is below 5%: 73 messages


**Goal.** Show what a random audit can and cannot tell.

**What we got.** An audit of 40 messages finds at least one of the 3 errors about half the time (52.5%), and when it finds none, the data still allow an error rate up to 8.8%. An audit of 20 would miss all three 70% of the time. It takes 73 messages with zero errors found to show, by the Wilson interval, that the error rate is under 5%. With a 1.6% true rate a sample of 75 comes back clean only 20% of the time, and one error in the sample moves the requirement to about 110 reads, two errors to about 142.

**What this shows for the pattern.** An audit exists to put an interval on the error rate of the messages nobody reads, so size it for the interval you need and not for the hours you have. For under 5% plan on 110 to 140 reads, the sample that still clears 5% after one or two errors; zero errors in all 183 acted messages would only reach 2.1%, so under 2% cannot be shown from this test set. Keep the sample random: reading only the low-confidence acted messages, as Part 1 does, finds more errors per hour but gives a biased estimate of the error rate of the whole stream.

## Part 3. A shadow run tells reviewers which messages to label

A candidate build runs beside the live one on live traffic. The live build's answers are used; the candidate's are logged and compared. Here the live head is bge-small and the candidate is the modernbert head from notebook 01, both scored on the 200 test messages. Messages where the two agree and are both right add nothing to the comparison, and messages where they agree and are both wrong cannot separate them. Only the messages where they disagree need a label.

In [4]:
mod_E = m.cached_embeddings("modernbert")
cand = LogisticRegression(C=r01["modernbert"]["C"], max_iter=3000).fit(mod_E["train"], ytr).predict(mod_E["test"])
live = top
disagree = np.where(cand != live)[0]
b = int(((live == yte) & (cand != yte)).sum())        # live right, candidate wrong
c = int(((cand == yte) & (live != yte)).sum())        # candidate right, live wrong
neither = int(((live != yte) & (cand != yte) & (cand != live)).sum())
p = binomtest(b, b + c, 0.5).pvalue
print(f"the two builds disagree on {len(disagree)} of {n} messages ({len(disagree) / n:.0%})")
print(f"  live right only: {b} | candidate right only: {c} | both wrong with different labels: {neither}")
print(f"  exact McNemar on the {b + c} messages where exactly one is right: p = {p:.3f}")
print(f"reviewers labeled {len(disagree)} messages instead of {n}")

the two builds disagree on 16 of 200 messages (8%)
  live right only: 12 | candidate right only: 3 | both wrong with different labels: 1
  exact McNemar on the 15 messages where exactly one is right: p = 0.035
reviewers labeled 16 messages instead of 200


**Goal.** Count how many messages a person must label to decide which build is better.

**What we got.** The builds disagree on 16 of 200 messages (8%). On those 16, the live build is right and the candidate wrong on 12, the candidate is right and the live build wrong on 3, and on 1 both are wrong with different labels. The exact McNemar test on the 15 messages where one is right gives p = 0.035, the same figure as in notebook 04, so a person labeling 16 messages reaches the conclusion that 200 labels gave.

**What this shows for the pattern.** The agreement rate of a shadow run is free and the disagreement set is the review queue: label only those messages, then test the difference. Two limits apply. A shadow run cannot find errors both builds make alike (2 of the live build's errors here are shared by the candidate), which is why Part 2's random audit stays. And 16 messages from one test set are a small sample: on live traffic, wait for enough disagreements that the test has power, and record the decision-logic version of both builds with each logged answer.

## Part 4. What the review tool must enforce

A review tool that shows the model's answer first invites the reviewer to confirm it. In a randomized experiment (199 participants, a simulated aid at 75% accuracy), three designs that made people commit first, ask for the AI, or wait before it appeared cut over-reliance on wrong suggestions from 64% to 48% on one task, and the authors report that people rated those designs worst and that the benefit went to the participants most motivated to think (Buçinca, Malaya and Gajos, CSCW 2021; https://arxiv.org/abs/2102.09692). Nearly half of the reviewers still took the wrong answer, so the design lowers the problem and does not remove it.

The class below enforces four rules in code: the reviewer sees the message and the candidate routes but not the model's pick; a verdict needs a reason from a fixed list (otherwise a click costs nothing); "cannot tell" is a valid typed answer that is counted, not dropped; and a batch is merged only if every message sent came back exactly once.

In [5]:
CANNOT_TELL = "cannot_tell"
REASONS = ("message_wording", "booking_detail", "policy_text", "other")

class ReviewBatch:
    def __init__(self, items):                      # items: {message_id: {"text", "candidates", "model_route"}}
        self.items, self.verdicts, self.log = items, {}, []
    def show(self, mid):                            # what the reviewer sees: no model pick
        it = self.items[mid]; return {"text": it["text"], "candidates": it["candidates"]}
    def commit(self, mid, route, reason):
        if mid not in self.items: raise KeyError(f"unknown message {mid}")
        if mid in self.verdicts: raise ValueError(f"{mid} already has a verdict")
        if route not in m.ROUTES and route != CANNOT_TELL: raise ValueError(f"{route!r} is not a route or cannot_tell")
        if reason not in REASONS: raise ValueError(f"reason must be one of {REASONS}")
        self.verdicts[mid] = {"route": route, "reason": reason}
    def reveal(self, mid):                          # the model's pick, only after the reviewer committed
        if mid not in self.verdicts: raise PermissionError("commit a verdict before the model's answer is shown")
        return self.items[mid]["model_route"]
    def merge(self):
        missing = set(self.items) - set(self.verdicts)
        if missing: raise ValueError(f"batch incomplete: {len(missing)} of {len(self.items)} messages have no verdict")
        decided = {k: v for k, v in self.verdicts.items() if v["route"] != CANNOT_TELL}
        return {"decided": decided, "cannot_tell": [k for k, v in self.verdicts.items() if v["route"] == CANNOT_TELL]}

esc = np.where(~acted)[0]
batch = ReviewBatch({te[i]["id"]: {"text": te[i]["text"], "candidates": [m.ROUTES[j] for j in np.where(P[i] >= 1 - Q)[0]],
                                   "model_route": m.ROUTES[slow[i]]} for i in esc})
mid0 = te[esc[0]]["id"]
tests = []
try: batch.reveal(mid0)
except PermissionError: tests.append("reveal before commit refused")
try: batch.commit(mid0, "refund_claim", "free text")
except ValueError: tests.append("reason outside the list refused")
for i in esc:                                       # a scripted reviewer who always gives the gold label
    batch.commit(te[i]["id"], m.ROUTES[yte[i]], "message_wording")
try: batch.commit(mid0, "refund_claim", "other")
except ValueError: tests.append("second verdict for one message refused")
merged = batch.merge()
overturned = sum(v["route"] != batch.items[k]["model_route"] for k, v in merged["decided"].items())
print("; ".join(tests))
print(f"merged {len(merged['decided'])} verdicts, {len(merged['cannot_tell'])} cannot_tell | the scripted reviewer overturns the slow tier on {overturned} of {len(esc)}")

short = ReviewBatch(batch.items); short.verdicts = dict(list(batch.verdicts.items())[:-1])
try: short.merge()
except ValueError as e: print("truncated batch refused:", e)

reveal before commit refused; reason outside the list refused; second verdict for one message refused
merged 17 verdicts, 0 cannot_tell | the scripted reviewer overturns the slow tier on 2 of 17
truncated batch refused: batch incomplete: 1 of 17 messages have no verdict


**Goal.** Run the four rules on the 17 escalated messages and show each one refusing what it should.

**What we got.** The tool refuses a reveal before a verdict, a reason outside the list and a second verdict for one message, and it refuses a batch with one verdict missing ("16 of 17"). The scripted reviewer, who always matches the gold label, overturns the slow tier on 2 of 17; that is the most an ideal reviewer could overturn, and a real reviewer's overturn rate below that number means the reviewer sometimes agrees with a wrong answer. The completeness check mirrors a rule from a team that sent about 13,000 rows through model reviewers in 16 batches: a reviewer returns its flags plus the number of rows it read, and the merge refuses a file whose count differs from the batch, because a truncated answer would otherwise read as "no flags". The team's note, not a published result.

**What this shows for the pattern.** The review stage has three measurable quantities: errors found per 100 messages read (Part 1), the interval on the error rate of unread messages (Part 2), and the overturn rate against the model, which a team should track by reviewer and by week. An overturn rate near zero means either an excellent model or a reviewer who confirms, and only a random audit, or a few seeded messages with known answers, tells the two apart.

In [6]:
out = {"n": n, "acted": int(acted.sum()), "errors": int(wrong.sum()), "yield": yield_rows,
       "audit": audit_rows, "audit_min_k_zero_errors_below_5pct": k,
       "shadow": {"disagree": int(len(disagree)), "live_only": b, "candidate_only": c, "neither": neither, "mcnemar_p": float(p)},
       "scripted_reviewer_overturns": int(overturned), "escalated": int(len(esc))}
(RESULTS / "13_human_review_stage.json").write_text(json.dumps(out, indent=2, default=float))

1298